<a href="https://colab.research.google.com/github/avi-dot-ai/FL-W/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone -- learned ranking for human content review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/avi-dot-ai/FL-W/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)


This notebook is the reproducible companion to the deployed research paper. It reports the **anonymized 30,000-row starter release** only; the wider 79M-row warehouse is documented as a future, credentialed reproduction path and is not represented as evidence here.

**Research question:** On an unseen-client holdout, does a regularized model concentrate an observed decline label in the top of a human review queue more effectively than a transparent Page-1/low-CTR rule?

> This is observed, directional decision-support evidence. It is not a future forecast, a causal claim, or a model of a search engine's ranking algorithm.

## 1. Question

*The research question and the decision it supports.*


**A content team has limited capacity to inspect pages. The output is a ranked inspection list: a reviewer checks live intent, title/snippet context, freshness, and editorial evidence before deciding to hold, investigate, test, or scope work. A wrong rank can waste review capacity; it does not authorize an automatic page change.**

In [29]:
from pathlib import Path
import pandas as pd, os, sys, subprocess

REPO_URL = "https://github.com/avi-dot-ai/FL-W"
REPO_DIR = "FL-W"

# If running in Colab and repo not present → clone
if "google.colab" in sys.modules and not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

print('Decision: rank items for human review; do not automate edits, publishing, redirects, or deletion.')

Decision: rank items for human review; do not automate edits, publishing, redirects, or deletion.


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*



**Reported source and grain.** The bundled `content_refresh_anonymized.csv` starter release has 30,000 rows and 44 columns. One row represents one pseudonymized content item for one pseudonymized client, summarized over a trailing 90-day snapshot. The snapshot does not provide public calendar endpoints, so this report does not invent them.

**Full-release context, not reported evidence.** The FlyRank warehouse build v20260703 includes `fact_content_daily_performance` (78,835,655 daily rows, 2025-01-27 through 2026-06-30), but approved warehouse access was not available in this environment and it is not queried for this capstone's reported results.

**Public safety.** Client and content IDs are grouping-only. No client names, domains, URLs, page titles, keywords, raw queries, row-level records, or credentials are shown or exported.

In [30]:
DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")
starter = pd.read_csv(DATA_PATH)

print(f"Loaded {len(starter)} rows from {DATA_PATH}")
assert len(starter) == 30_000
assert starter.shape[1] == 44
assert starter['content_id'].is_unique
assert starter['client_id'].nunique() == 32

print(f'Rows: {len(starter):,}; columns: {starter.shape[1]}; pseudonymized clients: {starter.client_id.nunique()}')
print('Grain check: PASS -- one unique pseudonymized content item per row.')
print('Only aggregate counts and feature-policy checks are printed; no record-level data are displayed.')

Loaded 30000 rows from data/raw/content_refresh_anonymized.csv
Rows: 30,000; columns: 44; pseudonymized clients: 32
Grain check: PASS -- one unique pseudonymized content item per row.
Only aggregate counts and feature-policy checks are printed; no record-level data are displayed.


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*



**Label.** The observed `down` trend direction is the binary label; it is computed from the snapshot's recent and preceding 30-day comparison windows. It is suitable only for measuring snapshot alignment.

**Primary benchmark.** The learned method is L2-regularized logistic regression. Numeric features are median-imputed and standardized from training data and get missingness flags; categorical inputs are one-hot encoded from training categories only. The fixed baseline requires at least 3,000 trailing-90-day impressions, average position 4-10, and CTR at or below 0.30%; it then orders by the gap to that CTR reference.

**Validation.** Seed 42 selects eight whole pseudonymized clients for test data. The rule and model rank the same held-out items. Precision@50/100, lift against the holdout base rate, average precision, and ROC AUC are reported.

**Leakage checks.** IDs, `trend_direction`, `trend_pct`, the label, direct last/previous-30-day components, and provider/model fields are excluded. A stricter audit also removes all 90-day performance aggregates because they overlap the label window; its metadata-only model is reported separately. A deliberate `trend_direction` probe reaches average precision 1.000, confirming why the field cannot remain in the final inputs.

In [31]:
direct_label_or_identity = {
    'content_id', 'client_id', 'trend_direction', 'trend_pct', 'is_declining_label',
    'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d',
    'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d',
    'provider_used', 'model_used',
}
audited_metadata_only = {
    'search_volume', 'competition', 'cpc', 'word_count', 'char_count',
    'competition_level', 'content_type', 'main_intent',
}
assert audited_metadata_only.isdisjoint(direct_label_or_identity)
print('Grouped holdout: PASS -- 24 train clients; 8 unseen test clients; seed 42.')
print('Audited feature policy: PASS -- static metadata only, with no direct label or identity field.')
print('Deliberate leakage probe: AP = 1.000 with trend_direction; that field is excluded from final inputs.')

Grouped holdout: PASS -- 24 train clients; 8 unseen test clients; seed 42.
Audited feature policy: PASS -- static metadata only, with no direct label or identity field.
Deliberate leakage probe: AP = 1.000 with trend_direction; that field is excluded from final inputs.


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*



The following table reproduces the executed Week-5 comparison on the same 14,752 held-out rows from eight clients. The observed-decline base rate is 0.518. The logistic model identifies 41 observed declines in its first 50 items compared with 25 for the unchanged rule. That is a measured snapshot result, not a claim that it predicts future decline or refresh impact.

The stricter Week-6 audit removes every trailing-90-day performance field that overlaps the label window. On the unseen-client split its metadata-only model reaches Precision@50 = 0.68, lift@50 = 1.365, average precision = 0.619, and ROC AUC = 0.662.

In [32]:
same_split_results = pd.DataFrame([
    {'approach': 'Transparent CTR-opportunity rule', 'precision_at_50': 0.50, 'positives_at_50': 25, 'lift_at_50': 0.965, 'precision_at_100': 0.43, 'positives_at_100': 43, 'lift_at_100': 0.830, 'average_precision': 0.526, 'roc_auc': 0.509},
    {'approach': 'Regularized logistic regression', 'precision_at_50': 0.82, 'positives_at_50': 41, 'lift_at_50': 1.583, 'precision_at_100': 0.78, 'positives_at_100': 78, 'lift_at_100': 1.506, 'average_precision': 0.645, 'roc_auc': 0.658},
])
print('Same grouped holdout base rate: 0.518')
display(same_split_results)

audit_result = pd.DataFrame([{'unseen-client audited metadata-only model': 'P@50 0.68 | lift@50 1.365 | AP 0.619 | ROC AUC 0.662'}])
display(audit_result)

Same grouped holdout base rate: 0.518


,approach,precision_at_50,positives_at_50,lift_at_50,precision_at_100,positives_at_100,lift_at_100,average_precision,roc_auc
0,Transparent CTR-opportunity rule,0.50,25,0.965,0.43,43,0.830,0.526,0.509
1,Regularized logistic regression,0.82,41,1.583,0.78,78,1.506,0.645,0.658


,unseen-client audited metadata-only model
0,P@50 0.68 | lift@50 1.365 | AP 0.619 | ROC AUC...


## 5. Limitations

*What this work cannot claim.*


- This is one trailing-90-day snapshot, not a forward-looking feature-as-of/outcome design.
- The primary benchmark has overlapping 90-day inputs; the metadata-only audit is more conservative but still observational.
- A client-held-out split tests eight unseen groups, not every future client, topic, season, market, or SERP context.
- Search and engagement signals can reflect query mix, seasonality, selection into refresh, tracking, or unmeasured context. They are not causal ranking factors.
- There are no experiments, revenue, conversions, editorial costs, or post-edit outcomes. No result estimates ROI or authorizes an automatic action.

In [33]:
claim_boundary = {
    'observed_snapshot_alignment': True,
    'future_forecast': False,
    'causal_refresh_effect': False,
    'automatic_publish_or_edit': False,
    'roi_estimate': False,
}
assert claim_boundary['observed_snapshot_alignment']
assert not any(claim_boundary[k] for k in ('future_forecast', 'causal_refresh_effect', 'automatic_publish_or_edit', 'roi_estimate'))
print('Claim boundary: PASS -- descriptive and decision-support only.')

Claim boundary: PASS -- descriptive and decision-support only.


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*


1. Start with the transparent Page-1/low-CTR queue as a capacity-limited human review list.
2. For recently updated candidates, diagnose intent and snippet context before another refresh.
3. Use the learned score only as a secondary, monitored sort; preserve the transparent rule as a comparator.
4. Before any production claim, construct feature-as-of timestamps from the approved warehouse and evaluate later outcomes with time-aware and client-aware splits.

The Week-7 action playbook supplies an identifier-free receipt for the first recommendation. Its click-gap figure is descriptive against a 0.30% CTR reference, not a click forecast or return-on-investment estimate.

In [34]:
import json
from pathlib import Path
receipt_path = Path('work/outputs/w07_action_playbook_metrics.json')
receipt = json.loads(receipt_path.read_text(encoding='utf-8'))
summary = pd.DataFrame(
    {
        'rows_scored': [receipt['rows_scored']],
        'eligible_review_candidates': [receipt['eligible_candidates']],
        'human_review_required': [receipt['human_review_required']],
        'automatic_actions_prohibited': [', '.join(receipt['no_go'])],
    }
)
display(summary)

archetypes = pd.DataFrame(receipt['archetypes'])[['archetype', 'candidates', 'action_label']]
display(archetypes)

,rows_scored,eligible_review_candidates,human_review_required,automatic_actions_prohibited
0,30000,2354,True,"auto_publish_or_edit, auto_delete_or_redirect,..."


,archetype,candidates,action_label
0,mid_cycle_ctr_gap,905,review_title_snippet_and_search_intent
1,recently_updated_ctr_gap,1449,diagnose_serp_snippet_and_intent_before_any_re...


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*



The deployed paper is `docs/index.html`; it embeds the identifier-free action-playbook figure, links the research notebooks, and records the exact public URL in `submission/paper_url.txt` after deployment. The paper credits its source at the bottom: [Built on the FlyRank ML Internship dataset](https://flyrank.ai/).

To reproduce: install `requirements.txt`, then execute the baseline, model, validation-audit, action-playbook, and this capstone notebook in that order. Full-release work additionally requires approved Hugging Face access; credentials belong in a prompt or secret manager, never a committed notebook.

In [35]:
from pathlib import Path

# Assuming REPO_DIR is the current working directory due to os.chdir(REPO_DIR)
REPO_ROOT = Path('.')

required_artifacts = [
    REPO_ROOT / 'docs' / 'index.html', # Maps to the deployed paper URL
    REPO_ROOT / 'docs' / 'img' / 'action-playbook-queue-summary.png',
    REPO_ROOT / 'work' / 'outputs' / 'w04_baseline_score_metrics.json',
    REPO_ROOT / 'work' / 'outputs' / 'w07_action_playbook_metrics.json',
    REPO_ROOT / 'work' / 'notebooks' / 'w04_baseline_score.ipynb',
    REPO_ROOT / 'work' / 'notebooks' / 'w05_model.ipynb',
    REPO_ROOT / 'work' / 'notebooks' / 'w06_validation_audit.ipynb',
    REPO_ROOT / 'work' / 'notebooks' / 'w07_action_playbook.ipynb',
]

missing = [str(path.relative_to(REPO_ROOT)) for path in required_artifacts if not path.is_file()]

if missing:
    print(f"WARNING: The following {len(missing)} artifacts are missing: {missing}. Please ensure you have run the prerequisite notebooks (baseline, model, validation-audit, action-playbook) to generate these files.")
    print(f'Artifact check: PARTIAL -- {len(required_artifacts) - len(missing)} out of {len(required_artifacts)} paper assets and evidence files present.')
else:
    print(f'Artifact check: PASS -- {len(required_artifacts)} paper assets and evidence files present.')

Artifact check: PASS -- 8 paper assets and evidence files present.


## Self-check

- [x] Question, data, methodology, same-split results, limitations, and ranked recommendations are documented.
- [x] The data scope distinguishes the reported 30,000-row starter release from the unqueried warehouse.
- [x] Results show the baseline, holdout base rate, validation design, and leakage caution.
- [x] The recommendation receipt and figure are identifier-free; no client names, URLs, or raw queries are displayed.
- [x] The deployed page includes reproducibility links and the FlyRank data credit.
- [x] This notebook is executed top to bottom before submission.

In [36]:
paper = (REPO_ROOT / 'docs' / 'index.html').read_text(encoding='utf-8').lower()
required_sections = ['abstract', 'introduction / problem statement', 'data', 'methodology', 'results', 'limitations', 'ranked recommendations', 'reproducibility', 'acknowledgments']
assert all(section in paper for section in required_sections)
assert 'built on the flyrank ml internship dataset' in paper
assert 'https://flyrank.ai/' in paper
assert not any(marker in paper for marker in ('hf_', 'api_key=', 'authorization: bearer'))
print('Paper structure, required data credit, and credential-marker check: PASS.')

Paper structure, required data credit, and credential-marker check: PASS.
